In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

DB_PATH = Path("../backtest_results.db")
con = sqlite3.connect(DB_PATH)

df_sltp = pd.read_sql("SELECT * FROM f1_session ORDER BY return_pct DESC", con)
df_ce   = pd.read_sql("SELECT * FROM f1_session_chandelier ORDER BY return_pct DESC", con)
con.close()

df_sltp["strategy"] = "DEMA_SL_TP"
df_ce["strategy"]   = "Chandelier_Exit"

In [2]:
metrics = ["return_pct", "sharpe_ratio", "profit_factor", "win_rate", "max_dd", "expectancy", "num_trades"]

summary = pd.DataFrame({
    "DEMA_SL_TP (best)": df_sltp[metrics].iloc[0],
    "Chandelier_Exit (best)": df_ce[metrics].iloc[0],
})
summary["ratio (SL_TP / CE)"] = summary["DEMA_SL_TP (best)"] / summary["Chandelier_Exit (best)"]
print("=== Best-run comparison ===")
print(summary.to_string())

=== Best-run comparison ===
               DEMA_SL_TP (best)  Chandelier_Exit (best)  ratio (SL_TP / CE)
return_pct              0.014930                0.005796            2.575786
sharpe_ratio            0.177943                0.074013            2.404210
profit_factor           1.022929                1.009357            1.013446
win_rate               41.436170               37.738483            1.097982
max_dd                  0.030839                0.039749            0.775835
expectancy              0.003971                0.001349            2.944342
num_trades           1880.000000             2149.000000            0.874826


In [3]:
print(f"\nDEMA SL/TP:  {(df_sltp['return_pct'] > 0).sum()} / {len(df_sltp)} positive combos ({(df_sltp['return_pct'] > 0).mean()*100:.1f}%)")
print(f"Chandelier:  {(df_ce['return_pct'] > 0).sum()} / {len(df_ce)} positive combos ({(df_ce['return_pct'] > 0).mean()*100:.1f}%)")


DEMA SL/TP:  19 / 27000 positive combos (0.1%)
Chandelier:  4 / 16200 positive combos (0.0%)


In [4]:
agg_metrics = ["return_pct", "sharpe_ratio", "profit_factor", "win_rate", "max_dd", "expectancy"]

combined = pd.concat([df_sltp, df_ce], ignore_index=True)
print("\n=== Aggregate stats (mean / median / std) ===")
for m in agg_metrics:
    g = combined.groupby("strategy")[m].agg(["mean", "median", "std"])
    print(f"\n--- {m} ---")
    print(g.to_string())


=== Aggregate stats (mean / median / std) ===

--- return_pct ---
                     mean    median       std
strategy                                     
Chandelier_Exit -1.015443 -0.386956  1.421950
DEMA_SL_TP      -1.305723 -0.479520  1.963985

--- sharpe_ratio ---
                      mean    median        std
strategy                                       
Chandelier_Exit -12.760143 -5.716542  15.806871
DEMA_SL_TP      -17.758355 -7.171472  24.540479

--- profit_factor ---
                     mean    median       std
strategy                                     
Chandelier_Exit  0.620125  0.660926  0.163319
DEMA_SL_TP       0.585692  0.639170  0.206582

--- win_rate ---
                      mean     median        std
strategy                                        
Chandelier_Exit  40.901960  40.516854   9.108229
DEMA_SL_TP       45.831213  46.742489  10.409407

--- max_dd ---
                     mean    median       std
strategy                                     
Chande

In [5]:
print("\n=== Best params: DEMA SL/TP ===")
print(df_sltp.iloc[0].to_string())
print("\n=== Best params: Chandelier Exit ===")
print(df_ce.iloc[0].to_string())


=== Best params: DEMA SL/TP ===
id                             19318
n1                                 5
atr_period                        18
sl_mult                          3.0
rr_ratio                         2.0
use_vwap                           1
commission                   0.00015
smooth_period                     12
asset                         XAUUSD
timeframe                        M15
session                       LONDON
expectancy                  0.003971
win_rate                    41.43617
max_dd                      0.030839
sharpe_ratio                0.177943
profit_factor               1.022929
num_trades                      1880
return_pct                   0.01493
run_timestamp    2026-08-24 08:48:15
strategy                  DEMA_SL_TP

=== Best params: Chandelier Exit ===
id                         12598
n1                             7
atr_period                    14
sl_mult                      3.0
trail_mult                   3.0
use_vwap                

In [ ]:
common_cols = ["timeframe", "session"]
for col in common_cols:
    if col in df_sltp.columns and col in df_ce.columns:
        print(f"\n=== Return by {col} ===")
        s1 = df_sltp.groupby(col)["return_pct"].agg(["mean", "median", "max"])
        s1.columns = [f"SLTP_{c}" for c in s1.columns]
        s2 = df_ce.groupby(col)["return_pct"].agg(["mean", "median", "max"])
        s2.columns = [f"CE_{c}" for c in s2.columns]
        print(s1.join(s2).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("DEMA SL/TP vs Chandelier Exit", fontsize=14)

for ax, m, title in zip(
    axes.flat,
    ["return_pct", "sharpe_ratio", "profit_factor", "expectancy"],
    ["Total Return %", "Sharpe Ratio", "Profit Factor", "Expectancy"],
):
    ax.hist(df_sltp[m].dropna(), bins=50, alpha=0.6, label="DEMA SL/TP")
    ax.hist(df_ce[m].dropna(), bins=50, alpha=0.6, label="Chandelier Exit")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()